# Terminal Centroids, Voronoi Regions, and Interactive OD Matrix

Notebook extended with an interactive Transfer List, dynamic Voronoi recomputation, row-normalized OD heatmap, Top N bidirectional flow lines, and descending activity ID renumbering.

In [38]:
import html
from pathlib import Path

import branca.colormap as cm
import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.spatial import Voronoi
from shapely import wkt
from shapely.affinity import rotate
from shapely.geometry import MultiPoint, Polygon, box
from sklearn.cluster import DBSCAN

pd.set_option("display.max_columns", 120)

In [39]:
DATA_FILE = Path("data/2024_11_17_17_00_processed.csv")
SEGMENTS_FILE = Path("data/data_2024-11-17.csv")

PARAMS = {
    "time_gap_new_trip_s": 5 * 60,
    "max_plausible_speed_kmh": 180,
    "stop_speed_kmh": 3,
    "long_stop_s": 3 * 60,
    "endpoint_cluster_eps_m": 150,
    "endpoint_cluster_min_samples": 8,
    "min_trip_points": 4,
    "min_trip_duration_s": 30,
    "min_trip_distance_m": 200,
    "bike_max_reported_speed_kmh": 35,
    "bike_max_gps_speed_kmh": 45,
    "bike_median_speed_kmh": 25,
    "car_min_p90_speed_kmh": 45,
    "car_min_max_speed_kmh": 60,
    "voronoi_clip_rotation_deg": 45,
    "voronoi_clip_padding_m": 600,
}

USE_BASIC_TRIP_FILTERS = True
USE_CAR_LIKE_FILTER = True

In [40]:
df_raw = pd.read_csv(DATA_FILE, parse_dates=["timestamp", "time"])
df_raw = df_raw.sort_values(["vehicle_id", "timestamp"]).copy()

segments_raw = pd.read_csv(SEGMENTS_FILE, parse_dates=["time"])
segments_raw = segments_raw.rename(columns={"segmnet_id": "segment_id"})
segments_daily = (
    segments_raw
    .groupby(["segment_id", "wkt"], as_index=False)
    .agg(
        avg_speed_day=("avg_speed", "mean"),
        median_speed_day=("avg_speed", "median"),
        observations=("avg_speed", "size"),
    )
)
segments_daily["geometry"] = segments_daily["wkt"].apply(wkt.loads)
segments_gdf = gpd.GeoDataFrame(segments_daily, geometry="geometry", crs="EPSG:4326")

In [41]:
def haversine_m(lat1, lon1, lat2, lon2):
    radius_m = 6_371_000
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * radius_m * np.arcsin(np.sqrt(a))

df = df_raw.copy()
df["prev_timestamp"] = df.groupby("vehicle_id")["timestamp"].shift(1)
df["prev_lat"] = df.groupby("vehicle_id")["lat"].shift(1)
df["prev_lon"] = df.groupby("vehicle_id")["lon"].shift(1)
df["time_diff_s"] = (df["timestamp"] - df["prev_timestamp"]).dt.total_seconds()
df["distance_m"] = haversine_m(df["prev_lat"], df["prev_lon"], df["lat"], df["lon"])
df.loc[df["time_diff_s"].isna(), "distance_m"] = np.nan
df["gps_speed_kmh"] = (df["distance_m"] / df["time_diff_s"]) * 3.6

df["is_stationary_point"] = df["speed"].le(PARAMS["stop_speed_kmh"])
df["is_implausible_jump"] = df["gps_speed_kmh"].gt(PARAMS["max_plausible_speed_kmh"])

df["stationary_block"] = df.groupby("vehicle_id")["is_stationary_point"].transform(lambda s: s.ne(s.shift()).cumsum())
stop_blocks = df[df["is_stationary_point"]].groupby(["vehicle_id", "stationary_block"]).agg(stop_start=("timestamp", "min"), stop_end=("timestamp", "max")).reset_index()
stop_blocks["stop_duration_s"] = (stop_blocks["stop_end"] - stop_blocks["stop_start"]).dt.total_seconds()
long_stops = stop_blocks[stop_blocks["stop_duration_s"] >= PARAMS["long_stop_s"]].copy()

long_stop_keys = set(zip(long_stops["vehicle_id"], long_stops["stationary_block"]))
df["is_long_stop_block"] = list(zip(df["vehicle_id"], df["stationary_block"]))
df["is_long_stop_block"] = df["is_long_stop_block"].isin(long_stop_keys)
df["prev_is_long_stop_block"] = df.groupby("vehicle_id")["is_long_stop_block"].shift(1, fill_value=False)

df["new_trip_reason"] = "continue"
df.loc[df["time_diff_s"].isna(), "new_trip_reason"] = "first_point"
df.loc[df["time_diff_s"].gt(PARAMS["time_gap_new_trip_s"]), "new_trip_reason"] = "time_gap"
df.loc[df["is_implausible_jump"], "new_trip_reason"] = "gps_jump"
df.loc[df["prev_is_long_stop_block"] & ~df["is_long_stop_block"], "new_trip_reason"] = "after_long_stop"
df["new_trip"] = df["new_trip_reason"].ne("continue")
df["trip_seq"] = df.groupby("vehicle_id")["new_trip"].cumsum()
df["trip_uid"] = df["vehicle_id"].astype(str) + "_" + df["trip_seq"].astype(str)

In [42]:
trip_stats = (
    df.groupby("trip_uid")
    .agg(
        vehicle_id=("vehicle_id", "first"),
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
        points=("timestamp", "size"),
        start_lat=("lat", "first"),
        start_lon=("lon", "first"),
        end_lat=("lat", "last"),
        end_lon=("lon", "last"),
        reported_speed_mean=("speed", "mean"),
        reported_speed_median=("speed", "median"),
        reported_speed_p90=("speed", lambda s: s.quantile(0.90)),
        reported_speed_max=("speed", "max"),
        gps_speed_max=("gps_speed_kmh", "max"),
        distance_m=("distance_m", "sum"),
        implausible_jumps=("is_implausible_jump", "sum"),
    )
    .reset_index()
)
trip_stats["duration_s"] = (trip_stats["end_time"] - trip_stats["start_time"]).dt.total_seconds()
trip_stats["valid_basic"] = ~(trip_stats["points"].lt(PARAMS["min_trip_points"]) | trip_stats["duration_s"].lt(PARAMS["min_trip_duration_s"]) | trip_stats["distance_m"].lt(PARAMS["min_trip_distance_m"]) | trip_stats["implausible_jumps"].gt(0))

car_like = trip_stats["reported_speed_p90"].ge(PARAMS["car_min_p90_speed_kmh"]) | trip_stats["reported_speed_max"].ge(PARAMS["car_min_max_speed_kmh"])
trip_stats["mode_class"] = np.select([car_like], ["car_like"], default="other")

endpoint_source_mask = trip_stats["valid_basic"] & trip_stats["mode_class"].eq("car_like")
trip_stats_filtered = trip_stats[endpoint_source_mask].copy()

In [43]:
start_points = trip_stats_filtered[["trip_uid", "start_lat", "start_lon"]].rename(columns={"start_lat": "lat", "start_lon": "lon"})
start_points["endpoint_type"] = "origin"
end_points = trip_stats_filtered[["trip_uid", "end_lat", "end_lon"]].rename(columns={"end_lat": "lat", "end_lon": "lon"})
end_points["endpoint_type"] = "destination"
endpoints = pd.concat([start_points, end_points], ignore_index=True).dropna(subset=["lat", "lon"])

In [44]:
eps_rad = PARAMS["endpoint_cluster_eps_m"] / 6_371_000
coords_rad = np.radians(endpoints[["lat", "lon"]].to_numpy())
clusterer = DBSCAN(eps=eps_rad, min_samples=PARAMS["endpoint_cluster_min_samples"], metric="haversine")
endpoints["endpoint_cluster"] = clusterer.fit_predict(coords_rad)

In [45]:
cluster_stats = (
    endpoints[endpoints["endpoint_cluster"] >= 0]
    .groupby("endpoint_cluster")
    .agg(
        lat=("lat", "mean"),
        lon=("lon", "mean"),
        events=("trip_uid", "size"),
        origins=("endpoint_type", lambda s: (s == "origin").sum()),
        destinations=("endpoint_type", lambda s: (s == "destination").sum())
    )
    .reset_index()
)

# SORTOWANIE: Sortujemy po liczbie zdarzeń malejąco, ale ZACHOWUJEMY oryginalne ID
cluster_stats = cluster_stats.sort_values("events", ascending=False).reset_index(drop=True)
cluster_stats["terminal_id"] = "T" + cluster_stats["endpoint_cluster"].astype(str)

In [46]:
def voronoi_finite_polygons_2d(vor, radius=None):
    if vor.points.shape[1] != 2:
        raise ValueError("Voronoi requires 2D points")
    new_regions, new_vertices = [], vor.vertices.tolist()
    center = vor.points.mean(axis=0)
    if radius is None: radius = np.ptp(vor.points, axis=0).max() * 2
    all_ridges = {}
    for (p1, p2), (v1, v2) in zip(vor.ridge_points, vor.ridge_vertices):
        all_ridges.setdefault(p1, []).append((p2, v1, v2))
        all_ridges.setdefault(p2, []).append((p1, v1, v2))
    for p1, region_idx in enumerate(vor.point_region):
        vertices = vor.regions[region_idx]
        if all(v >= 0 for v in vertices):
            new_regions.append(vertices)
            continue
        ridges = all_ridges.get(p1, [])
        new_region = [v for v in vertices if v >= 0]
        for p2, v1, v2 in ridges:
            if v2 < 0: v1, v2 = v2, v1
            if v1 >= 0: continue
            tangent = vor.points[p2] - vor.points[p1]
            tangent /= np.linalg.norm(tangent)
            normal = np.array([-tangent[1], tangent[0]])
            midpoint = vor.points[[p1, p2]].mean(axis=0)
            direction = np.sign(np.dot(midpoint - center, normal)) * normal
            new_region.append(len(new_vertices))
            new_vertices.append((vor.vertices[v2] + direction * radius).tolist())
        vertices_arr = np.asarray([new_vertices[v] for v in new_region])
        centroid = vertices_arr.mean(axis=0)
        angles = np.arctan2(vertices_arr[:, 1] - centroid[1], vertices_arr[:, 0] - centroid[0])
        new_regions.append([v for _, v in sorted(zip(angles, new_region))])
    return new_regions, np.asarray(new_vertices)

## Dynamic Recomputation & Analysis Functions

In [47]:
ACTIVE_TERMINALS = set()

def get_active_terminals():
    return cluster_stats[cluster_stats["endpoint_cluster"].isin(ACTIVE_TERMINALS)].copy()

# Base boundaries for clipping regions
base_terminals_2180 = gpd.GeoDataFrame(cluster_stats, geometry=gpd.points_from_xy(cluster_stats["lon"], cluster_stats["lat"]), crs="EPSG:4326").to_crs(epsg=2180)
segments_2180 = segments_gdf.to_crs(epsg=2180)
angle, padding = PARAMS["voronoi_clip_rotation_deg"], PARAMS["voronoi_clip_padding_m"]

clip_source_geom = base_terminals_2180.geometry.union_all().union(segments_2180.geometry.union_all())
center = clip_source_geom.centroid
rotated_source = rotate(clip_source_geom, -angle, origin=center)
minx, miny, maxx, maxy = rotated_source.bounds
rotated_clip_rect = box(minx - padding, miny - padding, maxx + padding, maxy + padding)
clip_geom = rotate(rotated_clip_rect, angle, origin=center)

def rebuild_voronoi():
    active_terminals = get_active_terminals()
    if len(active_terminals) < 3:
        raise ValueError("Voronoi requires at least 3 active terminals.")
        
    terminals_gdf_dynamic = gpd.GeoDataFrame(active_terminals, geometry=gpd.points_from_xy(active_terminals.lon, active_terminals.lat), crs="EPSG:4326")
    terminals_2180 = terminals_gdf_dynamic.to_crs(epsg=2180)
    points_xy = np.column_stack([terminals_2180.geometry.x, terminals_2180.geometry.y])
    
    vor = Voronoi(points_xy)
    regions, vertices = voronoi_finite_polygons_2d(vor, radius=100_000)
    
    polygons = []
    for region in regions:
        polygon = Polygon(vertices[region])
        if not polygon.is_valid: polygon = polygon.buffer(0)
        polygons.append(polygon.intersection(clip_geom))
        
    voronoi_dynamic = terminals_2180.copy()
    voronoi_dynamic["geometry"] = polygons
    voronoi_dynamic = gpd.GeoDataFrame(voronoi_dynamic, geometry="geometry", crs="EPSG:2180")
    voronoi_dynamic = voronoi_dynamic[~voronoi_dynamic.geometry.is_empty].copy()
    voronoi_dynamic["area_km2"] = voronoi_dynamic.geometry.area / 1_000_000
    
    return voronoi_dynamic.to_crs(epsg=4326)


In [48]:
def get_od_flows(voronoi_dynamic, trips):
    orig_gdf = gpd.GeoDataFrame(trips, geometry=gpd.points_from_xy(trips.start_lon, trips.start_lat), crs="EPSG:4326")
    dest_gdf = gpd.GeoDataFrame(trips, geometry=gpd.points_from_xy(trips.end_lon, trips.end_lat), crs="EPSG:4326")
    
    orig_assigned = gpd.sjoin(orig_gdf, voronoi_dynamic[["terminal_id", "geometry"]], how="inner", predicate="within")
    dest_assigned = gpd.sjoin(dest_gdf, voronoi_dynamic[["terminal_id", "geometry"]], how="inner", predicate="within")
    
    od_pairs = orig_assigned[["trip_uid", "terminal_id"]].rename(columns={"terminal_id": "origin_terminal"})
    od_pairs = od_pairs.merge(dest_assigned[["trip_uid", "terminal_id"]].rename(columns={"terminal_id": "destination_terminal"}), on="trip_uid")
    
    flows = od_pairs.groupby(["origin_terminal", "destination_terminal"]).size().reset_index(name="trips")
    return flows

def plot_dynamic_voronoi(voronoi_dynamic, flows_df, top_n=5):
    colormap = cm.LinearColormap(
        colors=["#440154", "#3b528b", "#21918c", "#5ec962", "#fde725"],
        vmin=voronoi_dynamic["events"].min(),
        vmax=voronoi_dynamic["events"].max()
    )
    colormap.caption = "Terminal Activity (number of events)"

    m = folium.Map(location=[52.25, 20.98], zoom_start=11, tiles="CartoDB positron")
    
    folium.GeoJson(
        voronoi_dynamic,
        name="Dynamic Voronoi",
        style_function=lambda feature: {
            "fillColor": colormap(feature["properties"]["events"]),
            "color": "#333333",
            "weight": 1.2,
            "fillOpacity": 0.35,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["terminal_id", "events", "origins", "destinations", "area_km2"],
            aliases=["Terminal:", "Events:", "Origins:", "Destinations:", "Area (km2):"]
        )
    ).add_to(m)

    term_coords = voronoi_dynamic.set_index('terminal_id')[['lat', 'lon']].to_dict('index')
    
    # Calculate Bidirectional Flows
    f = flows_df.copy()
    f['term1'] = np.minimum(f['origin_terminal'], f['destination_terminal'])
    f['term2'] = np.maximum(f['origin_terminal'], f['destination_terminal'])
    
    bi_flows = f.groupby(['term1', 'term2'])['trips'].sum().reset_index()
    # Remove self-loops (internal terminal trips)
    bi_flows = bi_flows[bi_flows['term1'] != bi_flows['term2']]
    top_flows = bi_flows.nlargest(top_n, 'trips')
    
    if not top_flows.empty:
        max_trips = top_flows['trips'].max()
        for _, row in top_flows.iterrows():
            t1, t2, w = row['term1'], row['term2'], row['trips']
            c1 = (term_coords[t1]['lat'], term_coords[t1]['lon'])
            c2 = (term_coords[t2]['lat'], term_coords[t2]['lon'])
            
            line_weight = 2 + 10 * (w / max_trips)
            
            folium.PolyLine(
                locations=[c1, c2], 
                color="#ff2a00", 
                weight=line_weight, 
                opacity=0.6,
                tooltip=f"{t1} &harr; {t2}: <b>{w}</b> total trips"
            ).add_to(m)
    
    for _, row in voronoi_dynamic.iterrows():
        cluster_num = str(row['terminal_id']).replace('T', '')
        folium.Marker(
            [row['lat'], row['lon']],
            icon=folium.DivIcon(
                html=f'<div style="font-size: 11px; font-weight: bold; color: #111; background: #ffd23f; border: 1px solid #111; border-radius: 50%; width: 22px; height: 22px; text-align: center; line-height: 20px; box-shadow: 1px 1px 3px rgba(0,0,0,0.3);">{cluster_num}</div>',
                icon_size=(22, 22),
                icon_anchor=(11, 11)
            )
        ).add_to(m)

    colormap.add_to(m)
    return m

def plot_od_heatmap(flows_df, unique_terms):
    # Sort terms numerically for heatmap axes so T2 comes before T10
    all_terms = sorted(unique_terms, key=lambda x: int(x.replace('T', '')))
    
    od_matrix = flows_df.pivot(index="origin_terminal", columns="destination_terminal", values="trips").fillna(0)
    od_matrix = od_matrix.reindex(index=all_terms, columns=all_terms, fill_value=0)
    
    row_sums = od_matrix.sum(axis=1).replace(0, 1)
    od_matrix_norm = od_matrix.div(row_sums, axis=0)
    
    fig, ax = plt.subplots(figsize=(14, 10))
    sns.heatmap(
        od_matrix_norm, 
        cmap="mako_r", 
        annot=False, 
        linewidths=0.5,
        cbar_kws={'label': 'Row-normalized flow'},
        ax=ax
    )
    ax.set_title("Row-Normalized OD Matrix (Origin-Destination)", fontsize=16)
    ax.set_ylabel("Origin Terminal", fontsize=12)
    ax.set_xlabel("Destination Terminal", fontsize=12)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    
    # Zamknięcie figury w pamięci matplotlib, aby Jupyter nie renderował jej podwójnie
    plt.close(fig) 
    return fig

## Dashboard Control Panel

In [49]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
plt.ioff()

# =========================================================
# PREPARE TERMINALS
# =========================================================

cluster_stats_sorted = (
    cluster_stats
    .sort_values("events", ascending=False)
    .reset_index(drop=True)
)

ALL_TERMINALS = [
    int(x)
    for x in cluster_stats_sorted["endpoint_cluster"]
    if x >= 0
]

events_map = (
    cluster_stats_sorted
    .set_index("endpoint_cluster")["events"]
    .to_dict()
)

def terminal_label(t):
    return f"T{t} ({events_map.get(t, 0)} trips)"

def sort_terminals(values):
    return sorted(
        list(set(values)),
        key=lambda x: ALL_TERMINALS.index(x)
    )

# =========================================================
# STATE
# =========================================================

active_terminals = sort_terminals(ALL_TERMINALS)
inactive_terminals = []

# =========================================================
# SELECTORS
# =========================================================

active_selector = widgets.SelectMultiple(
    options=[(terminal_label(t), t) for t in active_terminals],
    rows=15,
    layout=widgets.Layout(width="320px")
)

inactive_selector = widgets.SelectMultiple(
    options=[],
    rows=15,
    layout=widgets.Layout(width="320px")
)

# =========================================================
# BUTTONS
# =========================================================

btn_remove = widgets.Button(
    description=">>",
    button_style="warning",
    layout=widgets.Layout(width="70px")
)

btn_add = widgets.Button(
    description="<<",
    button_style="success",
    layout=widgets.Layout(width="70px")
)

btn_generate = widgets.Button(
    description="Generate Dashboard",
    icon="refresh",
    button_style="info",
    layout=widgets.Layout(width="250px")
)

# =========================================================
# HELPERS
# =========================================================

def refresh_selectors():

    active_selector.options = [
        (terminal_label(t), t)
        for t in sort_terminals(active_terminals)
    ]

    inactive_selector.options = [
        (terminal_label(t), t)
        for t in sort_terminals(inactive_terminals)
    ]

# =========================================================
# MOVE LOGIC
# =========================================================

def move_to_inactive(_):

    global active_terminals
    global inactive_terminals

    selected = list(active_selector.value)

    if len(selected) == 0:
        return

    # remove from active
    active_terminals = [
        t for t in active_terminals
        if t not in selected
    ]

    # add once only
    inactive_terminals = sort_terminals(
        inactive_terminals + selected
    )

    refresh_selectors()


def move_to_active(_):

    global active_terminals
    global inactive_terminals

    selected = list(inactive_selector.value)

    if len(selected) == 0:
        return

    inactive_terminals = [
        t for t in inactive_terminals
        if t not in selected
    ]

    active_terminals = sort_terminals(
        active_terminals + selected
    )

    refresh_selectors()

# =========================================================
# CALLBACKS
# =========================================================

btn_remove.on_click(move_to_inactive)
btn_add.on_click(move_to_active)

# =========================================================
# OUTPUTS
# =========================================================

map_output = widgets.Output(
    layout=widgets.Layout(
        width="60%",
        border="1px solid #ccc",
        padding="5px"
    )
)

matrix_output = widgets.Output(
    layout=widgets.Layout(
        width="40%",
        border="1px solid #ccc",
        padding="5px"
    )
)

status = widgets.HTML(
    value="<b>Status:</b> Ready"
)

# =========================================================
# DASHBOARD GENERATION
# =========================================================

is_running = False

current_map = None
current_fig = None

def generate_dashboard(_):

    global ACTIVE_TERMINALS
    global is_running
    global current_map
    global current_fig

    if is_running:
        return

    is_running = True
    btn_generate.disabled = True

    try:

        selected = active_terminals.copy()

        if len(selected) < 3:
            status.value = (
                "<b style='color:red;'>"
                "Choose at least 3 terminals"
                "</b>"
            )
            return

        # =================================================
        # CLEAN OLD OUTPUTS
        # =================================================

        map_output.clear_output(wait=True)
        matrix_output.clear_output(wait=True)

        if current_fig is not None:
            plt.close(current_fig)
            current_fig = None

        current_map = None

        # =================================================
        # REBUILD
        # =================================================

        ACTIVE_TERMINALS = set(selected)

        status.value = "<b>Status:</b> Rebuilding Voronoi..."

        vor_dyn = rebuild_voronoi()

        flows = get_od_flows(
            vor_dyn,
            trip_stats_filtered
        )

        # =================================================
        # MAP
        # =================================================

        current_map = plot_dynamic_voronoi(
            vor_dyn,
            flows,
            top_n=15
        )

        with map_output:
            clear_output(wait=True)
            display(current_map)

        # =================================================
        # MATRIX
        # =================================================

        current_fig = plot_od_heatmap(
            flows,
            vor_dyn["terminal_id"].unique()
        )

        with matrix_output:
            clear_output(wait=True)
            display(current_fig)

        plt.close(current_fig)

        status.value = (
            "<b style='color:green;'>"
            "Dashboard generated"
            "</b>"
        )

    except Exception as e:

        status.value = (
            f"<b style='color:red;'>Error: {e}</b>"
        )

    finally:

        is_running = False
        btn_generate.disabled = False


# =========================================================
# CONNECT BUTTON
# =========================================================

btn_generate.on_click(generate_dashboard)

# =========================================================
# UI
# =========================================================

transfer_panel = widgets.HBox([

    widgets.VBox([
        widgets.HTML("<b>Active terminals</b>"),
        active_selector
    ]),

    widgets.VBox([
        btn_remove,
        btn_add
    ],
    layout=widgets.Layout(
        justify_content="center",
        padding="20px"
    )),

    widgets.VBox([
        widgets.HTML("<b>Inactive terminals</b>"),
        inactive_selector
    ])

])

dashboard_panel = widgets.HBox([
    map_output,
    matrix_output
])

main_ui = widgets.VBox([

    widgets.HTML("<h2>Interactive OD Dashboard</h2>"),

    transfer_panel,

    btn_generate,

    status,

    dashboard_panel

])

# =========================================================
# DISPLAY
# =========================================================

clear_output(wait=True)

display(main_ui)
